# Tier-A launcher: EXP-A-001, one generator model per session

**Execution wrapper only.** All logic runs inside the repo
(`faithfulids.orchestration.cli run --experiment EXP-A-001`). Accelerator **GPU T4 x2**,
Internet **on**, input dataset **CICIDS2017** (`chethuhn/network-intrusion-dataset`).

Tier-A runs in **two steps**, because a generator model and the registered extractor
(Gemma-4-26B-A4B, 14.4 GB) do not fit one 2x T4 session together:

1. **`PHASE = 'generate'`** loads ONE generator (`MODEL`) and writes every explanation
   into a call ledger under `tier_a/ledgers/generation/<model>/`. No claims, no metrics,
   no run. It stops cleanly after `BUDGET_MIN` minutes; calls already in the ledger are
   never repeated, so the next session continues. Repeat until the log says
   **`generation COMPLETE`**.
2. **`PHASE = 'score'`** needs no generator. It serves Gemma with Ollama (amendment 0005),
   mints the EXP-G-001 token, runs EXP-G-002 (CPU, 30-60 min), then replays the
   generations, extracts claims with the certified extractor, computes the metrics and
   writes the Tier-A run.

**Carrying the ledgers between sessions.** A session starts empty. Upload the previous
session's `tier_a/ledgers/` as a private Kaggle dataset and add it as an input; the
restore cell finds any `ledgers/` folder under `/kaggle/input`.

**It stops on its own when it should:** the detector competence gate is enforced (the
pre-registered contingency, amendment 0001, applies first if a class fails it), and the
score step refuses to start without both gate tokens.

**Recorded gaps (written into every run's `instrument_gaps`):** B4/B5 use the rule
verifier (the registered phi verifier is not pinned); no plausibility judge, so H1 is
not scored; pilot-grade data cleaning.

`N = ''` is the registered N=400. A small `N` is a smoke test: it gets its own ledgers
and its run records the smaller N, so it can never pass for the Tier-A record.

In [ ]:
# Launcher parameters ONLY. Every token-spending choice is made here.
import os
PINNED_REF    = 'main'          # the terminal push replaces this with the exact commit
PHASE         = 'generate'      # 'generate' (repeat until COMPLETE) | 'score'
MODEL         = 'qwen3_8b_4bit' # ONE per session: qwen3_4b_4bit | qwen3_8b_4bit | qwen3_32b_4bit | mistral_7b_instruct_4bit
N             = ''              # '' = registered N=400 | small number = smoke test only
BUDGET_MIN    = '600'           # generate stops cleanly after this (Kaggle's hard limit is 12 h)
ROWS_PER_FILE = '50000'         # per-file stride cap so every day's attack families load; same in both steps

assert PHASE in ('generate', 'score'), PHASE
OUT = '/kaggle/working/tier_a'
TAG = MODEL + (f'_n{N}' if N else '')   # smoke ledgers never mix with the registered ones
os.makedirs(OUT, exist_ok=True)
os.environ.update({
    'FAITHFULIDS_REF': PINNED_REF,
    'FAITHFULIDS_PHASE': PHASE,
    'FAITHFULIDS_PILOT_LLM': MODEL,
    'FAITHFULIDS_ROWS_PER_FILE': ROWS_PER_FILE,
    'FAITHFULIDS_GENERATION_BUDGET_MIN': BUDGET_MIN,
    'FAITHFULIDS_LLM_CACHE_DIR': f'{OUT}/ledgers/generation/{TAG}',
    'FAITHFULIDS_EXTRACTION_CACHE_DIR': f'{OUT}/ledgers/extraction/{TAG}',
    'FAITHFULIDS_DEVICE_MAP': 'auto',   # the 32B generator (~18 GB in 4 bit) needs both T4s
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'TIER_A_OUT': OUT, 'TIER_A_TAG': TAG,
})
os.environ.pop('FAITHFULIDS_MAX_ROWS', None)   # a global cap keeps only the first day's file
if N:
    os.environ['FAITHFULIDS_PILOT_N'] = N
else:
    os.environ.pop('FAITHFULIDS_PILOT_N', None)
print(f"phase: {PHASE} | model: {MODEL} | N: {N or '400 (registered)'} | ref: {PINNED_REF}")

In [ ]:
%%bash
set -e
set -o pipefail   # each pip line pipes into `tail`: without this a failed install passes silently
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || {
  echo "NO GPU. Settings -> Accelerator -> 'GPU T4 x2'."; exit 1; }
[ "$(nvidia-smi --list-gpus | wc -l)" -ge 2 ] || echo "WARNING: one GPU only; the 32B generator and Gemma need both T4s."
# Clone OUTSIDE /kaggle/working: the saved output is only what tier_a/ collects.
rm -rf /tmp/faithful-ids
git clone --quiet https://github.com/SLIMIHamda/faithful-ids.git /tmp/faithful-ids
cd /tmp/faithful-ids && git checkout --quiet "$FAITHFULIDS_REF"
git rev-parse HEAD | tee "$TIER_A_OUT/PINNED_COMMIT_$FAITHFULIDS_PHASE.txt"
# --ignore-requires-python: Kaggle runs Python 3.13; pyproject's `<3.13` is for the pinned
# stack, which --no-deps already skips. The import below proves the install took.
pip install -q -e . --no-deps --ignore-requires-python 2>&1 | tail -n 2
pip install -q pyyaml jsonschema bitsandbytes accelerate 2>&1 | tail -n 2
if [ "$FAITHFULIDS_PHASE" = "generate" ]; then
  # Generator sessions pin transformers < 5 (the v5.0 bnb 4-bit load regression,
  # huggingface/transformers#43032). The score step loads no transformers model.
  pip install -q "transformers<5" 2>&1 | tail -n 2
fi
python -c "import faithfulids, xgboost, shap, sys; print('faithfulids', faithfulids.__version__, '| xgboost', xgboost.__version__, '| shap', shap.__version__, '| python', sys.version.split()[0])"

In [ ]:
# CICIDS2017 CSVs (auto-detected under /kaggle/input) and the HF token for gated models.
import glob
cands = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/*.csv', recursive=True)})
def has_label(d):
    f = sorted(glob.glob(os.path.join(d, '*.csv')))[:1]
    return f and 'label' in open(f[0]).readline().lower()
cands = [d for d in cands if has_label(d)]
assert cands, 'No CICIDS2017 CSVs under /kaggle/input: add chethuhn/network-intrusion-dataset as input.'
os.environ['FAITHFULIDS_DATA_DIR'] = cands[0]
print('CICIDS2017 dir:', cands[0], '|', len(glob.glob(os.path.join(cands[0], '*.csv'))), 'csv files')
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN'); print('HF_TOKEN set')
except Exception as e:
    print('No HF secret (fine unless the model is gated, e.g. Mistral):', type(e).__name__)

In [ ]:
# Restore the ledgers of earlier sessions (attached as an input dataset). For each
# model's ledger, the copy with the most calls wins.
import shutil
restored = 0
for kind in ('generation', 'extraction'):
    best = {}
    for p in glob.glob(f'/kaggle/input/**/ledgers/{kind}/*/ledger.jsonl', recursive=True):
        tag = os.path.basename(os.path.dirname(p))
        n = sum(1 for _ in open(p, encoding='utf-8'))
        if n > best.get(tag, (0, None))[0]:
            best[tag] = (n, p)
    for tag, (n, p) in best.items():
        dst = f'{OUT}/ledgers/{kind}/{tag}'
        os.makedirs(dst, exist_ok=True)
        shutil.copy(p, f'{dst}/ledger.jsonl')
        restored += 1
        print(f'restored {kind}/{tag}: {n} calls')
print('no earlier ledgers attached' if not restored else f'{restored} ledger(s) restored')
gen = f'{OUT}/ledgers/generation/{TAG}/ledger.jsonl'
if PHASE == 'score' and not os.path.isfile(gen):
    raise SystemExit(f'score needs the generation ledger for {TAG}: attach the dataset holding '
                     'tier_a/ledgers from the generate sessions')

In [ ]:
%%bash
# STEP 1 - generate (skipped in a score session).
[ "$FAITHFULIDS_PHASE" = "generate" ] || { echo "generate step skipped (PHASE=score)"; exit 0; }
set -o pipefail
cd /tmp/faithful-ids
python -m faithfulids.orchestration.cli run --experiment EXP-A-001 2>&1 \
  | tee "$TIER_A_OUT/generate_$TIER_A_TAG.log"
rc=$?
grep -h "^generation \(COMPLETE\|PARTIAL\)" "$TIER_A_OUT/generate_$TIER_A_TAG.log" \
  | tail -n 1 | tee "$TIER_A_OUT/GENERATION_STATUS_$TIER_A_TAG.txt"
# Exit 0 so the export cell still runs; it re-raises this code at the very end.
echo $rc > "$TIER_A_OUT/STEP_RC"

In [ ]:
# STEP 2a - serve the certified extractor (score session only): Google's QAT GGUF of
# gemma-4-26B-A4B at the pinned revision, sha256-checked, under Ollama (amendment 0005).
if PHASE == 'score':
    import hashlib, json, subprocess, time, urllib.request, yaml
    from huggingface_hub import hf_hub_download

    m = yaml.safe_load(open('/tmp/faithful-ids/configs/extraction/eval_extractor.yaml'))['model']
    w, ol = m['weights'], m['ollama']
    t0 = time.time()
    GGUF = hf_hub_download(w['hf_repo'], w['file'], revision=w['revision'], local_dir='/tmp/gguf')
    h = hashlib.sha256()
    with open(GGUF, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            h.update(chunk)
    assert h.hexdigest() == w['sha256'], f"sha256 {h.hexdigest()} is not the pinned {w['sha256']}"
    print(f"GGUF ok ({(time.time() - t0) / 60:.1f} min): sha256 {w['sha256'][:12]}")
    with open('/tmp/gguf/Modelfile', 'w') as f:
        f.write(f"FROM {GGUF}\nRENDERER {ol['renderer']}\nPARSER {ol['parser']}\n")

    subprocess.run('apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y zstd >/dev/null 2>&1', shell=True)
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
    env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_KEEP_ALIVE='-1', OLLAMA_NUM_PARALLEL='1')
    serve = subprocess.Popen(['ollama', 'serve'], env=env, stderr=subprocess.STDOUT,
                             stdout=open(f'{OUT}/ollama_serve.log', 'w'))
    for _ in range(60):
        try:
            ver = json.load(urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2))['version']
            break
        except Exception:
            time.sleep(2)
    else:
        raise SystemExit('ollama serve did not start: see tier_a/ollama_serve.log')
    need = tuple(int(x) for x in ol['min_version'].split('.'))
    assert tuple(int(x) for x in ver.split('-')[0].split('.')[:3]) >= need, f'Ollama {ver} is too old'
    open(f'{OUT}/OLLAMA_VERSION.txt', 'w').write(ver + '\n')
    subprocess.run(['ollama', 'create', ol['model_name'], '-f', '/tmp/gguf/Modelfile'], check=True)
    os.remove(GGUF)  # Ollama holds its own copy, stored under the same sha256
    req = urllib.request.Request('http://127.0.0.1:11434/api/chat', method='POST',
        headers={'Content-Type': 'application/json'},
        data=json.dumps({'model': ol['model_name'], 'stream': False, 'think': False,
                         'messages': [{'role': 'user', 'content': 'Reply with the single word OK.'}],
                         'options': {'temperature': 0, 'seed': 0, 'num_predict': 8, 'num_ctx': 8192}}).encode())
    print('warm-up:', json.load(urllib.request.urlopen(req, timeout=1800))['message']['content'])
    print(subprocess.run(['ollama', 'ps'], capture_output=True, text=True).stdout)
else:
    print('extractor setup skipped (PHASE=generate)')

In [ ]:
%%bash
# STEP 2b - gate tokens, then score (score session only). Both tokens land in THIS
# session's runs/, which is where the Tier-A check looks.
[ "$FAITHFULIDS_PHASE" = "score" ] || { echo "score step skipped (PHASE=generate)"; exit 0; }
set -o pipefail
cd /tmp/faithful-ids
export PYTHONPATH=src
KEY=$(python -c "import yaml; v = yaml.safe_load(open('configs/extraction/eval_extractor.yaml'))['version']; print('extractor_claims_' + v.replace('.', '_') + '_llm')")
echo "== EXP-G-001 token (GPU-free re-score of the committed audit) =="
python tools/score_audit_gate.py --batch experiments/gates/EXP-G-001_audit_v2 \
  --pass LLM_1_V2 --pass LLM_2_V2 --claims-key "$KEY" --write-run 2>&1 \
  | tee "$TIER_A_OUT/g001.log" | tail -n 3
rc=$?
if [ $rc -eq 0 ]; then
  echo "== EXP-G-002 (RQ0 calibration, CPU, registered N) =="
  env -u FAITHFULIDS_PILOT_N FAITHFULIDS_PILOT_DETECTOR=xgboost_multiclass \
    python -m faithfulids.orchestration.cli run --experiment EXP-G-002 2>&1 \
    | tee "$TIER_A_OUT/g002.log" | tail -n 3
  rc=$?
fi
if [ $rc -eq 0 ]; then
  echo "== EXP-A-001 score: $TIER_A_TAG =="
  python -m faithfulids.orchestration.cli run --experiment EXP-A-001 2>&1 \
    | tee "$TIER_A_OUT/score_$TIER_A_TAG.log"
  rc=$?
fi
# Exit 0 so the export cell still runs; it re-raises this code at the very end.
echo $rc > "$TIER_A_OUT/STEP_RC"

In [ ]:
%%bash
# Export whatever exists, success or not: ledgers (attach next session), runs, logs.
cd /tmp/faithful-ids 2>/dev/null && for d in EXP-A-001 EXP-G-001 EXP-G-002; do
  [ -d "runs/$d" ] && mkdir -p "$TIER_A_OUT/runs" && cp -r "runs/$d" "$TIER_A_OUT/runs/"
done
nvidia-smi > "$TIER_A_OUT/nvidia-smi.txt" 2>&1
cd /kaggle/working && rm -f tier_a_artifacts.zip && zip -qr tier_a_artifacts.zip tier_a
ls -la tier_a_artifacts.zip; du -sh tier_a/ledgers 2>/dev/null
cat tier_a/GENERATION_STATUS_*.txt 2>/dev/null || true
rc=$(cat tier_a/STEP_RC 2>/dev/null || echo 0)
[ "$rc" = 0 ] || echo "A step FAILED (rc=$rc): see the logs in tier_a/"
exit $rc